# Sanctions Name Screening project

## Project description:

Some text here about the project

Maybe some formulas 
$\frac{\sum}{\alpha}$

## Setup


In [ ]:
import sys; print(sys.executable)

In [ ]:
#imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import gc
import unicodedata
import sys
import pyarrow.parquet as pq
from rapidfuzz import fuzz
from rapidfuzz import distance as rf_distance
from sklearn.metrics import roc_auc_score

# Every path comes from src/screening/config.py, so this notebook and the
# feature scripts always read and write the same files. Works whether the
# notebook is started from notebooks/ or from the project root.
_here = Path.cwd().resolve()
sys.path.insert(0, str((_here.parent if _here.name == "notebooks" else _here) / "src"))
from screening import config as C


In [ ]:
project_root = C.ROOT
pairs_path = C.PAIRS_FLAT

pairs = pd.read_parquet(pairs_path)

print("Pairs dataset shape:", pairs.shape)


---

# What this notebook does

We ask each question related to Exploratory Data Analysis in a heading and write code below that answers it.

**Part 3** produces the cleaned table that parts
4 to 7 use.

When a question produces a number that will end up in the report, move that
code into `src/screening/` afterwards so it can be re-run. Notebooks explore,
scripts produce.

---

# 1. First look at the data

### 1.1 What shape is the data, and what does one row represent?

*Everything downstream depends on getting this sentence right. One row is one what?*

# `pairs`:

In [ ]:
pairs.head()

> The grain of `pairs` dataset is `pair_id` which represents the unique identifier of a pair of entities that an analyst reviewed after a generated potential match. Semantically, each row represents an analyst's decision on the entity.

# `names`:

In [ ]:
from collections import Counter

with open(C.DATA_RAW / "names.txt", encoding="utf-8", errors="ignore") as f:
    names = Counter(line.strip().casefold() for line in f if line.strip())

freq_names = pd.Series(names).sort_values(ascending=False)

print("Names frequency dataset shape:", freq_names.shape)
freq_names.head(20)

> `freq_names` is a Series which counts the frequency of names appearing in pairs. The grain is the unique name. However:
> - Tokens tell us which name-parts are common, which is more valuable for matching

In [ ]:
with open(C.DATA_RAW / "names.txt", encoding="utf-8", errors="ignore") as f:
    tokens = Counter(tok for line in f for tok in line.casefold().split())

freq_tokens = pd.Series(tokens).sort_values(ascending=False)
print("Tokens frequency dataset shape:", freq_tokens.shape)

freq_tokens.head()

In [ ]:
# Memory note. This notebook holds several large objects at once:
#   pairs        ~2.3 GB  (32 columns, most of them lists of strings)
#   freq_names   3.15M entries, built above and never used again
#   freq_tokens  1.6M entries, used later in 5.3 and 5.5
#
# If the kernel dies around here, that is why. Two ways out:
#   - drop freq_names, as below, and re-run
import gc
del freq_names
gc.collect()
print("freed freq_names; freq_tokens kept for 5.3 and 5.5")

> `freq_tokens` grain is unique token that appears in the `names` dataset

### 1.2 What record types are in the file, and are they all the same problem?

*Decides what stays in scope. Some pair types may not be name comparisons at all.*

In [ ]:
pairs.info()

Every entity property in this data is a **list of strings**, even when it holds a single value, because one record can carry several spellings of a name or several reported dates of birth. An absent field is an empty list, never a null. Code that assumes `value[0]` exists will fail on roughly half the rows for some fields.

| variable | meaning | unit | type | values/range | missing coding | role | status | notes/concerns |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| `pair_id` | the unique identifier of a pair of entities | - | integer `int64` | e.g. `1`, `2` | none present | **index** | keep | 
| `judgement` | analyst's decision about the pair | - | categorical `str` | `positive`, `negative` | none present | **target** | keep | documentation lists a third value, `unsure`, which does not occur in this file |
| `left_id` / `right_id` | OpenSanctions entity identifier | - | string `str`| e.g. `Q18224025`, `NK-aTXq...` | none present | identifier | keep, not a feature | 38,217 rows have the same id on both sides |
| `left_schema` / `right_schema` | kind of entity | - | categorical `str` | Person, Company, Organization, Occupancy, Succession, Position, ... | none present | filter + predictor | keep | used to scope the project to three types |
| `left_caption` / `right_caption` | display name | - | string `str` | 1-375 characters, median 23 | none present | main predictor source | keep | one of the names in `*_name`; using it alone discards the aliases |
| `left_name` / `right_name` | all name spellings | - | list of strings `object` | median 1 per entity, p90 12, max 293 | `[]`, 1.2% of pairs | predictor source | keep | comparing across full lists is more expensive but should beat caption-only |
| `left_alias` / `right_alias` | further alternative names | - | list of strings `object` | both sides present in 41.2% | `[]` | predictor source | keep | overlaps with `name`, needs de-duplicating |
| `left_birthDate` / `right_birthDate` | reported date(s) of birth | date | list of strings `object` | mixed precision: `1969`, `1969-06`, `1969-06-16` | `[]`, both sides present in 49.2% | strong predictor | keep | stored as text, not dates; 11.6% of entities carry more than one |
| `left_country` / `right_country` | associated countries | - | list of ISO-2 codes `object` | both sides present in 61.0% | `[]` | predictor | keep | agreement is weak evidence, disagreement is strong |
| `left_nationality` / `right_nationality` | nationality | - | list of ISO-2 codes `object` | both sides present in 15.9% | `[]` | predictor | keep with caution | too sparse to carry much weight |
| `left_idNumber` / `right_idNumber` | passport, tax, registration numbers | - | list of strings `object` | both sides present in 9.1% | `[]` | predictor | keep | decisive when present, almost never present |
| `left_firstName` / `left_lastName` / `left_middleName` | parsed name components | - | list of strings `object` | both sides present in 40-45% | `[]` | predictor | keep | parsing quality varies by source list |
| `left_address` / `right_address` | addresses | - | list of free text `object` | both sides present in 42.7% | `[]` | probably not used | park | unstructured; industry commentary blames free-text addresses for much of the false-positive problem, so it is interesting but out of scope for us |
| `left_topics` / `right_topics` | risk tags | - | list of strings `object` | `sanction`, `role.pep`, `crime.fin`, ... | `[]` | **excluded** | drop | describes the listing, not the identity |
| `left_datasets` / `right_datasets` | which source lists the record came from | - | list of strings `object` | 293 possible values | none present | **excluded, leaky** | drop | predicts the label at 92% accuracy on its own |

### 1.3 What does a single row actually contain?

*Look at one row in full. Note which fields are lists rather than single values.*

In [ ]:
pairs.iloc[1,:]

> One row represents characteristics of one entity

> Features, which are lists (or `objects`):

| feature that is a list | Dtype | 
|---|---|
| `left_datasets` |  `object`|
  | `right_datasets`|       `object`|
  |`left_name`   |         `object`|
  |`right_name`   |        `object`|
  |`left_alias`   |        `object`|
  |`right_alias`  |        `object`|
  |`left_firstName`|       `object`|
  |`right_firstName`|      `object`|
  |`left_lastName`  |      `object`|
 | `right_lastName`   |    `object`|
 | `left_middleName` |     `object`|
 | `right_middleName` |    `object`|
  |`left_birthDate`  |     `object`|
| `right_birthDate` |    `object`|
  |`left_country`   |      `object`|
 | `right_country` |       `object`|
 | `left_nationality` |    `object`|
 | `right_nationality` |   `object`|
  |`left_idNumber`  |      `object`|
  |`right_idNumber` |    `object` |  
  |`left_address` |        `object`|
  |`right_address` |       `object`|
 |`left_topics` |        `object`|
 | `right_topics`  |      `object`|

### 1.4 Which columns describe the entities, and which describe how the data was made?

*Flag the second kind now; they come back in part 7.*

In [ ]:
pairs.columns

In [ ]:
import pandas as pd
from pathlib import Path

df = pd.read_parquet(C.PAIRS_FLAT)

ROLE = {
    "pair_id":     ("our own key", "row number we assigned at parse time"),
    "judgement":   ("TARGET",      "the analyst's decision; this is the label"),
    "id":          ("identifier",  "names the record, not the entity's properties"),
    "schema":      ("entity",      "what kind of thing it is"),
    "caption":     ("entity",      "display name"),
    "name":        ("entity",      "all name spellings"),
    "alias":       ("entity",      "further names"),
    "firstName":   ("entity",      "parsed name part"),
    "middleName":  ("entity",      "parsed name part"),
    "lastName":    ("entity",      "parsed name part"),
    "birthDate":   ("entity",      "when the person was born"),
    "country":     ("entity",      "associated country"),
    "nationality": ("entity",      "nationality"),
    "idNumber":    ("entity",      "passport / tax / registration number"),
    "address":     ("entity",      "where they are"),
    "topics":      ("PROCESS",     "why the entity was listed, not who it is"),
    "datasets":    ("PROCESS",     "which source lists the record came from"),
}

rows = []
for col in df.columns:
    base = col.removeprefix("left_").removeprefix("right_")
    role, why = ROLE.get(base, ("?", "unclassified - look at it"))
    rows.append({"column": col, "base": base, "role": role, "why": why})

cols = pd.DataFrame(rows)

print(f"{len(df.columns)} columns, {cols.base.nunique()} distinct fields\n")
print(cols.drop_duplicates("base").role.value_counts().to_string())

print("\n--- columns that describe HOW THE DATA WAS MADE ---")
print(cols[cols.role == "PROCESS"][["column", "why"]].to_string(index=False))

print("\n--- identifiers: real, but never features ---")
print(cols[cols.role == "identifier"].column.tolist())

# Evidence, so the classification isn't just an assertion
print("\n`datasets` holds source-list names, nothing about the person:")
for c in df.left_datasets.head(4):
    print("   ", list(c))
print("\n`topics` holds listing reasons:")
for c in df.left_topics.head(4):
    print("   ", list(c))

### 1.5 What kind of identifier is on each side?

*Two kinds live in this file, and the difference explains most of what looks strange later.*

In [ ]:
# Two kinds of identifier live in this file, and the difference explains
# almost everything odd about the data.

import re

def id_kind(value):
    value = str(value)
    if value.startswith("NK-") or re.match(r"^Q\d+$", value):
        return "canonical"      # a merged entity, created by deduplication
    return "source"             # a raw record as one list published it

pairs["left_kind"] = pairs.left_id.map(id_kind)
pairs["right_kind"] = pairs.right_id.map(id_kind)

print("what a judgement actually compares:")
print((pairs.left_kind + " <-> " + pairs.right_kind).value_counts().to_string())

sides = pd.concat([
    pairs[["left_kind", "left_datasets"]].rename(
        columns={"left_kind": "kind", "left_datasets": "datasets"}),
    pairs[["right_kind", "right_datasets"]].rename(
        columns={"right_kind": "kind", "right_datasets": "datasets"}),
])
print("\nsource lists held per record:")
print(sides.groupby("kind").datasets.apply(
    lambda s: f"median {s.map(len).median():.0f}, p90 {s.map(len).quantile(.9):.0f}"
).to_string())

print("""
HOW THE DATA IS BUILT

OpenSanctions ingests records from 293 lists. Each raw record describes one
entity as a single list sees it, carries that list's own id, and holds exactly
one source.

The same person appears on many lists, so the records have to be merged. A
candidate generator proposes pairs, an analyst decides, and a "same" decision
merges the records into a CANONICAL entity with a new id. That entity
inherits the union of everything: every name spelling, every source, every
birth date. Hence the higher source count above.

Most judgements therefore ask: does this raw record belong to that existing
canonical entity?
""")


# And the part that matters for modelling: the id types predict the label.
combo = pd.Series(np.where(
    pairs.left_id == pairs.right_id, "X + X (same id)",
    np.where(pairs.left_kind == pairs.right_kind,
             pairs.left_kind + " + " + pairs.right_kind,
             "canonical + source")))

table = pd.crosstab(combo, pairs.judgement)
table["total"] = table.sum(axis=1)
table["pct_positive"] = (100 * table.positive / table.total).round(1)
print("\nlabel balance by id-type combination:")
print(table.sort_values("total", ascending=False).to_string())

print("""
THE LEAK HIDING HERE

Those percentages are nothing like each other, and they have nothing to do
with names:

  canonical + source   ~97% positive   routine: attach a new record to a
                                       known entity, almost always yes
  canonical + canonical ~8% positive   rare audit: did we accidentally create
                                       the same entity twice? almost always no
  source + source       ~1% positive   two raw records proposed to each other

The id type is telling us WHERE IN THE WORKFLOW the pair was shown to an
analyst, not whether two people are the same. Knowing only the id type gets
you most of the label.

So `left_id` and `right_id` are excluded from features in every form: not the
id, not its prefix, not its length, not a "is this a merged entity" flag. They
are identifiers and workflow metadata, never evidence about the entities.
Question 7.1 measures exactly how strong this is.
"""
)


### 1.6 What does a self-pair represent?

*Rows where both sides carry the same id. Work out what they are before deciding what to do with them.*

In [ ]:
# A self-pair is a row where both sides carry the SAME id. Before deciding
# what to do with them, work out what they represent.

self_pairs = pairs.left_id == pairs.right_id

print(f"self-pairs: {self_pairs.sum():,} rows ({100*self_pairs.mean():.1f}%)")
print("\nwhat kind of id do they carry?")
print(pairs.loc[self_pairs, "left_kind"].value_counts().to_string())
print("\nand ordinary pairs, for comparison:")
print(pairs.loc[~self_pairs, "left_kind"].value_counts().to_string())

print("\nsource lists held:")
for label, mask in [("self-pair", self_pairs), ("ordinary ", ~self_pairs)]:
    counts = pairs.loc[mask, "left_datasets"].map(len)
    print(f"  {label}: median {counts.median():.0f}, p90 {counts.quantile(.9):.0f}, "
          f"{100*(counts > 3).mean():.1f}% hold more than three")

example = pairs[self_pairs & (pairs.left_datasets.map(len) > 3)].iloc[0]
print(f"\nexample: {example.left_id}  '{example.left_caption}'")
print(f"  sources: {list(example.left_datasets)}")
print(f"  names  : {len(example.left_name)} spellings")

print("""
WHAT THEY ARE

Every self-pair carries a canonical id and never a source id, and they hold
far more sources than ordinary records. A record assembled from eleven lists
cannot be a raw extract; it is the product of a merge.

So the sequence was: an analyst compared two SOURCE records and said "same
entity"; the system merged them into one canonical entity; and when this file
was exported, both sides of that old judgement resolved to the merged id. The
row became (X, X).

That is why 38,819 of 38,830 are positive: the merge happened BECAUSE the
judgement was positive. The 11 negatives are the interesting ones - a record
declared not to be itself - and they are the cleanest evidence that these
labels are operational decisions rather than ground truth.

A self-pair teaches a model nothing: two identical records, answer always yes.
They are removed in 3.2.
""")

### 1.7 Is the identifier convention documented, or did we guess it?

*A rule we invented from looking at strings is a guess. Check the publisher's own documentation, then test whether the data could tell us the same thing without it.*


In [ ]:
# In 1.5 we classified each id by its shape. That rule has to come from
# somewhere better than our own pattern-matching, so: what does the publisher
# actually say, and could the data tell us the same thing on its own?

# The publisher's documentation (https://www.opensanctions.org/docs/identifiers)
# states that entities merged by OpenSanctions receive an identifier beginning
# with "NK-", and that where a merged entity is matched to Wikidata the
# Wikidata QID ("Q" followed by digits) is used as the canonical identifier
# instead. Records as published by an individual source keep an identifier
# prefixed with that source's dataset name.
#
# So the rule is documented, not invented. But a documented convention can
# still be violated in a 700k-row export, so we check every id against it.

import re

def id_kind(value):
    value = str(value)
    if value.startswith("NK-") or re.match(r"^Q\d+$", value):
        return "merged"
    return "source"

all_ids = pd.unique(pd.concat([pairs.left_id.astype(str),
                               pairs.right_id.astype(str)]))

shape = pd.Series(all_ids).map(
    lambda v: "NK-..."              if v.startswith("NK-")
    else      "Q + digits"         if re.match(r"^Q\d+$", v)
    else      "<dataset>-..."      if re.match(r"^[a-z][a-z0-9_]*-", v)
    else      "UNRECOGNISED"
)

print(f"distinct ids: {len(all_ids):,}\n")
print(shape.value_counts().to_string())

# Two things worth confirming rather than assuming.

# (a) Are there ids that look like Wikidata QIDs but do not match ^Q\d+$?
#     Those would be silently classified as "source" and corrupt the counts.
qish = [v for v in all_ids if str(v).upper().startswith("Q")
        and not re.match(r"^Q\d+$", str(v))]
print(f"\nQ-like but not ^Q\\d+$ : {len(qish)}   {qish[:5]}")

# (b) What exactly falls outside every pattern?
mask = (shape == "UNRECOGNISED").to_numpy()
unknown = pd.Series(all_ids)[mask]
print(f"\nunrecognised: {len(unknown)}")
for v in unknown.head(3):
    print("   ", v)
print("   (all share one prefix; they are source-side ids whose dataset name")
print("    contains a digit, so the fallback branch classifies them correctly)")

del all_ids, shape, mask
gc.collect()


---

# 2. Can we trust the labels?

Before measuring anything, find out what is wrong with the labels themselves.

### 2.1 Are there pairs where a record is paired with itself?

*A record trivially matches itself. If these exist they are free correct answers and would inflate every score.*

In [ ]:
self_pairs = pairs.left_id == pairs.right_id

print(f"Rows where left_id == right_id: {self_pairs.sum():,}"
      f"({100 * self_pairs.mean():.1f}% of {len(pairs):,})")

print("\nhow are they labelled?")
print(pairs.loc[self_pairs, "judgement"].value_counts().to_string())

print("\nwhat kind of records?")
print(pairs.loc[self_pairs, "left_schema"].value_counts().to_string())

print("\nare the captions identical too?")
same_caption = pairs.loc[self_pairs, "left_caption"] == pairs.loc[self_pairs, "right_caption"]
print(f"    identical caption: {same_caption.sum():,} of {self_pairs.sum():,} ({100 * same_caption.mean():.1f}%)")

print("\nfive examples:")
print(pairs.loc[self_pairs, ["left_id", "right_id", "left_caption", "right_caption", "judgement"]]
        .head().to_string(index=False))


- The contradiction worth capturing: **self-pairs labelled `negative`**:

- This simply cannot be true since pairs match in the list, while the analyst's decision was `negative`

In [ ]:
contradictory_pairs = pairs[self_pairs & (pairs.judgement == 'negative')]
print(f"\nself-pairs labelled NEGATIVE: {len(contradictory_pairs)}")
print(contradictory_pairs[["left_id", "right_id", "left_caption", "right_caption", "left_schema", "right_schema"]].to_string(index=False))

### 2.2 Are there duplicate pairs?

*Duplicates put identical examples in both train and test, so the test score partly measures memorisation.*

In [ ]:
mask = pairs.duplicated(
    subset=["left_id", "right_id"],
    keep=False
)

# Get duplicated rows
duplicated_pairs = pairs[mask]

# Count
print("Duplicated rows:", len(duplicated_pairs))

# Display
display(duplicated_pairs.head(10))



### 2.3 Do any repeated pairs disagree with each other on the label?

*If the same pair is labelled both ways, there is a ceiling on measurable accuracy that belongs to the labels, not the model.*

In [ ]:
conflict_mask = (
    duplicated_pairs
    .groupby(["left_id", "right_id"])["judgement"]
    .transform("nunique") > 1
)

different_judgements = duplicated_pairs[conflict_mask]

print(different_judgements[["left_id", "right_id", "judgement"]].to_string(index=False))

### 2.4 How often does the same entity appear across different pairs?

*Decides whether train/test can be split by row or must be split by entity.*

In [ ]:
appearances = pd.concat([pairs.left_id, pairs.right_id]).value_counts()

print(f"distinct entities: {len(appearances):,}")
print(f"median {appearances.median():.0f}, p99 {appearances.quantile(0.99):.0f}, "
      f"max {appearances.max():,}")

repeated = appearances[appearances > 1]
print(f"appear in more than one pair: {len(repeated):,} "
      f"({100 * len(repeated) / len(appearances):.1f}%)")

# the number that actually decides the split strategy
touches_repeated = (pairs.left_id.isin(repeated.index)
                    | pairs.right_id.isin(repeated.index))
print(f"\nrows touching a repeated entity: {touches_repeated.sum():,} "
      f"({100 * touches_repeated.mean():.1f}% of all rows)")

print("\nten most reused entities:")
print(appearances.head(10).to_string())

### 2.5 Are there missing or impossible values in the key columns?

*Nulls in ids, captions or the label. Negative or absurd values anywhere.*

In [ ]:
key_cols = ["pair_id", "judgement", "left_id", "right_id",
            "left_schema", "right_schema", "left_caption", "right_caption"]

print("nulls in key columns:")
print(pairs[key_cols].isna().sum().to_string())

print("\nempty strings (not null, but useless):")
for c in ["left_caption", "right_caption"]:
    print(f"  {c}: {(pairs[c].fillna('').str.strip() == '').sum():,}")

print("\njudgement takes these values:")
print(pairs.judgement.value_counts(dropna=False).to_string())

# the docs mention a third value, 'unsure'. Is it here?
print(f"\n'unsure' present? {(pairs.judgement == 'unsure').sum()}")

print("\ncaption length:")
lens = pairs.left_caption.fillna("").str.len()
print(f"  min {lens.min()}, median {lens.median():.0f}, max {lens.max()}")
print("\nthe three longest captions:")
for c in pairs.left_caption.fillna("").sort_values(key=lambda s: s.str.len()).tail(3):
    print(f"  [{len(c)} chars] {c[:110]}...")

---

# 3. Clean and prepare

Everything from here on uses the table this part produces. We write down the
reason for each decision in `docs/decisions.md` as we make it.

---

## 3.0 Check hypotheses before filtering

Filtering throws away 42% of rows. Before we do that, we need to prove they are extra, not just inconvenient.


### 3.0a Which record types carry names at all?
*The correct criteria is not the list of types, but presence of `name` feature.

In [ ]:
# HYPOTHESIS: some record types are not name comparisons at all,
# so including them would mean averaging over two different tasks.
# TEST: does the record carry a `name` property, or only a caption?

rows = []
for schema, g in pairs.groupby("left_schema"):
    rows.append({
        "schema": schema,
        "pairs": len(g),
        "has caption %": round(100 * (g.left_caption.fillna("").str.strip() != "").mean(), 1),
        "has name %":    round(100 * (g.left_name.map(len) > 0).mean(), 1),
        "example caption": str(g.left_caption.iloc[0])[:44],
    })

by_schema = pd.DataFrame(rows).sort_values("pairs", ascending=False)
print(by_schema.to_string(index=False))

named    = by_schema[by_schema["has name %"] > 50].schema.tolist()
unnamed  = by_schema[by_schema["has name %"] <= 50].schema.tolist()

print(f"\ncarry a real name : {named}")
print(f"carry NO name     : {unnamed}")
print("\nLook at the example captions for the second group: the caption is")
print("literally the type's own word ('Occupancy', 'Family', 'Other link').")
print("These are relationship records - X held position Y, A is related to B.")
print("There is no name to compare, so they are a different task and go.")

### 3.0b Is the record type informative enough?

*Hypothesis: "a person cannot be a vessel - is a useful feature."*

In [ ]:
# HYPOTHESIS: type mismatch is a useful feature - "a person cannot be a vessel".
# TEST 1: do cross-type pairs like Person/Vessel even exist?

combo = pairs.left_schema + " / " + pairs.right_schema
cross = pairs.left_schema != pairs.right_schema

print(f"pairs with different types on each side: {cross.sum():,} "
      f"({100 * cross.mean():.1f}%)")
print("\nevery cross-type combination that occurs:")
print(combo[cross].value_counts().to_string())

print("\n--- the specific case: Person against a non-Person ---")
pv = pairs[((pairs.left_schema == "Person") & (pairs.right_schema != "Person"))
           | ((pairs.right_schema == "Person") & (pairs.left_schema != "Person"))]
print(f"rows: {len(pv):,}")
print("their partner types:")
partners = pd.concat([
    pv.loc[pv.left_schema == "Person", "right_schema"],
    pv.loc[pv.right_schema == "Person", "left_schema"],
])
print(partners.value_counts().to_string())

print("\nPerson vs Vessel specifically:",
      int(((pairs.left_schema == "Person") & (pairs.right_schema == "Vessel")).sum()
          + ((pairs.left_schema == "Vessel") & (pairs.right_schema == "Person")).sum()))
print("\nThe candidate generator blocks by type BEFORE proposing a pair to an")
print("analyst. It never asks whether a man is a ship. So a 'types differ'")
print("feature has no training examples of that kind - it would be a constant.")

# TEST 2: where cross-type pairs DO exist, does the mismatch predict anything?
from sklearn.metrics import roc_auc_score

print("\n--- but where they do exist, is the mismatch informative? ---")
print(pd.crosstab(cross, pairs.judgement, normalize="index").round(3).to_string())
auc = roc_auc_score((pairs.judgement == "positive").astype(int),
                    (~cross).astype(int))
print(f"\nAUC of the feature 'types match': {auc:.3f}")
print("0.5 is a coin toss. The feature carries nothing, so it is not a reason")
print("to keep or drop any record type.")

### 3.0c What a narrow filter costs?

*3 types against 7, and the cost of the decision.*

In [ ]:
# HYPOTHESIS: restricting to Person/Company/Organization is the right scope.
# TEST: what does the narrower choice actually cost us?

NARROW = ("Person", "Company", "Organization")
BROAD  = ("Person", "Company", "Organization", "LegalEntity",
          "Vessel", "Airplane", "PublicBody")

def clean_size(allowed):
    """Rows surviving the full cleaning pipeline under a given scope."""
    m = pairs.left_schema.isin(allowed) & pairs.right_schema.isin(allowed)
    s = pairs[m]
    ns = s[s.left_id != s.right_id].copy()
    ns["a"] = np.minimum(ns.left_id.astype(str), ns.right_id.astype(str))
    ns["b"] = np.maximum(ns.left_id.astype(str), ns.right_id.astype(str))
    c = ns[~ns.duplicated(subset=["a", "b"], keep="first")]
    return m.sum(), len(c), 100 * (c.judgement == "positive").mean()

for label, allowed in [("narrow (3 types)", NARROW), ("broad (7 types)", BROAD)]:
    in_scope_n, clean_n, pos = clean_size(allowed)
    print(f"{label:<18} in scope {in_scope_n:>7,}  ->  clean {clean_n:>7,}  "
          f"({pos:.1f}% positive)")

m_n = pairs.left_schema.isin(NARROW) & pairs.right_schema.isin(NARROW)
m_b = pairs.left_schema.isin(BROAD) & pairs.right_schema.isin(BROAD)
extra = pairs[m_b & ~m_n]

print(f"\nwhat the narrow scope throws away: {len(extra):,} rows")
print((extra.left_schema + " / " + extra.right_schema).value_counts().to_string())

both_named = (extra.left_name.map(len) > 0) & (extra.right_name.map(len) > 0)
print(f"\nof those, carry a name on BOTH sides: {100 * both_named.mean():.1f}%")
print(f"label balance of the discarded rows  : "
      f"{extra.judgement.value_counts(normalize=True).round(3).to_dict()}")

print("""
VERDICT
  These are real name comparisons, not relationship records. LegalEntity is
  the parent type of Company and Organization, so excluding it while keeping
  its children is arbitrary. Vessels and aircraft are genuinely screened in
  sanctions compliance.

  The defensible rule is not a hand-picked list of three types but: keep every
  record type that carries a name, drop the relationship records that do not.

  Cost of being broad: the added rows are heavily positive, so the overall
  balance shifts by about two points. Saying so in the report.
""")

---

### 3.1 Apply the scope filter

*Keep only the record types that are genuinely name comparisons. Record why.*

In [ ]:
IN_SCOPE = ("Person", "Company", "Organization")

# Filter on BOTH sides. Filtering only on left_schema leaves cross-type pairs
# such as Organization/LegalEntity and gives a different row count.
in_scope = pairs.left_schema.isin(IN_SCOPE) & pairs.right_schema.isin(IN_SCOPE)
scoped = pairs[in_scope].copy()

print(f"{len(pairs):,} -> {len(scoped):,} in scope "
      f"({100 * in_scope.mean():.1f}%)")

print("\nwhat we dropped, and why it is not a name comparison:")
dropped = pairs[~in_scope]
print((dropped.left_schema + " / " + dropped.right_schema)
      .value_counts().head(8).to_string())

### 3.2 Remove self-pairs and duplicates

*Keep a count of what each step removed; those numbers go in the report.*

In [ ]:
before = len(scoped)

# 1. Self-pairs first. They duplicate each other heavily, so counting
#    duplicates before removing them measures the wrong thing.
self_mask = scoped.left_id == scoped.right_id
no_self = scoped[~self_mask].copy()
n_self = int(self_mask.sum())

# 2. Normalise the pair order BEFORE de-duplicating.
#    left/right carries no meaning in this data, so (A,B) and (B,A) are the
#    same judgement recorded twice. Dropping duplicates on the raw columns
#    misses every reversed pair.
no_self["pair_a"] = np.minimum(no_self.left_id.astype(str),
                               no_self.right_id.astype(str))
no_self["pair_b"] = np.maximum(no_self.left_id.astype(str),
                               no_self.right_id.astype(str))

# 3. Pairs recorded with BOTH labels. Keeping the first copy would let the row
#    order in the file pick the label. The true answer is unknown, so every
#    copy of such a pair is removed.
key = ["pair_a", "pair_b"]
contradicting = no_self.groupby(key).judgement.transform("nunique") > 1
n_conflict_pairs = int(no_self.loc[contradicting, key].drop_duplicates().shape[0])
n_conflict_rows = int(contradicting.sum())
agreeing = no_self[~contradicting]

# 4. What is left are copies that agree on the label, so keeping the first
#    one loses nothing.
dup_mask = agreeing.duplicated(subset=key, keep="first")
clean = agreeing[~dup_mask].drop(columns=key).reset_index(drop=True)
n_dups = int(dup_mask.sum())

# Remember the counts, then free the intermediate. Each of these frames is
# over a gigabyte; holding four at once is what runs a laptop out of memory.
n_raw, n_scoped = len(pairs), len(scoped)
ordered_only = int(agreeing.duplicated(subset=["left_id", "right_id"],
                                       keep="first").sum())
del no_self, agreeing, dup_mask, contradicting
gc.collect()

print(f"in scope          : {before:,}")
print(f"  - self-pairs    : {n_self:,}")
print(f"  - contradicting : {n_conflict_rows:,} rows ({n_conflict_pairs:,} pairs labelled both ways)")
print(f"  - extra copies  : {n_dups:,}")
print(f"= clean           : {len(clean):,}")

print(f"\nIgnoring pair order finds {n_dups:,} duplicates.")
print(f"Not normalising the order would find only {ordered_only:,}, "
      f"missing {n_dups - ordered_only:,} reversed pairs.")
print(f"\nCounting duplicates BEFORE removing self-pairs gives "
      f"{scoped.duplicated(subset=['left_id','right_id'], keep=False).sum():,} rows,")
print("which is 99% self-pairs duplicating each other, not real duplicates.")

### 3.2b Do any negative pairs contradict a chain of positive pairs?

In [ ]:
# If analysts said A = B and B = C, then A and C are the same entity. A
# negative row A != C contradicts that chain. We count these but KEEP them:
# we cannot tell which of the three decisions is wrong, and removing the
# negative would quietly favour the positive ones.
# Groups are built exactly as in the split (part 8): positive rows are links.

from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

_l = clean.left_id.astype(str)
_r = clean.right_id.astype(str)
_ids = pd.Index(pd.unique(pd.concat([_l, _r])))
_li, _ri = _ids.get_indexer(_l), _ids.get_indexer(_r)
_pos = (clean.judgement == "positive").to_numpy()
_graph = coo_matrix((np.ones(int(_pos.sum())), (_li[_pos], _ri[_pos])),
                    shape=(len(_ids), len(_ids)))
_, _group = connected_components(_graph, directed=False)

neg_in_group = (~_pos) & (_group[_li] == _group[_ri])
n_neg_in_group = int(neg_in_group.sum())
print(f"negative pairs inside one positive-only group: {n_neg_in_group:,} (kept)")
print(clean.loc[neg_in_group, ["left_caption", "right_caption"]].head(5).to_string(index=False))
del _l, _r, _ids, _li, _ri, _pos, _graph, _group

### 3.3 What survived, and what is the label balance now?

*The balance will have moved. Report the cleaned figure, not the raw one.*

In [ ]:
balance = clean.judgement.value_counts()
print("label balance after cleaning:")
print((balance / len(clean) * 100).round(1).to_string())
print(f"\ncounts: {balance.to_dict()}")

raw_pos = 100 * (pairs.judgement == "positive").mean()
new_pos = 100 * (clean.judgement == "positive").mean()
print(f"\npositive share: {raw_pos:.1f}% raw -> {new_pos:.1f}% cleaned")
print("The raw figure was inflated by self-pairs, which are positive by")
print("definition. We report the cleaned one.")

print("\nrecord types remaining:")
print((clean.left_schema + " / " + clean.right_schema).value_counts().to_string())

### 3.4 Save the cleaned table

*Write it to data/interim/. Everything below reads this file, so nobody re-derives it differently.*

In [ ]:
out_path = C.PAIRS_CLEAN
out_dir = out_path.parent
out_dir.mkdir(parents=True, exist_ok=True)

clean.to_parquet(out_path, index=False)
print(f"wrote {out_path}  ({out_path.stat().st_size / 1e6:.0f} MB, {len(clean):,} rows)")

# A note that travels with the file, so the next person knows what is in it.
note = f"""# pairs_clean.parquet

Built by notebooks/02-vs-eda-cleaning-split.ipynb part 3 from data/interim/pairs_flat.parquet.

{len(pairs):,} raw -> {len(scoped):,} in scope -> {len(clean):,} usable
Label balance: {100 * (clean.judgement == 'positive').mean():.1f}% positive

Removed:
- record types that are not name comparisons (Occupancy, Succession, Position,
  Vessel, LegalEntity ...), filtered on BOTH sides
- {n_self:,} self-pairs (same id on both sides; positive by definition)
- {n_conflict_pairs:,} pairs recorded with both labels ({n_conflict_rows:,} rows):
  every copy removed, because the true label is unknown
- {n_dups:,} duplicate id-pairs (same label; the order of the two sides is
  normalised first, so (A,B) and (B,A) count as one pair)

Counted but KEPT:
- {n_neg_in_group:,} negative pairs whose two ids are joined by a chain of
  positive pairs (A=B, B=C, but A!=C). We cannot tell which decision is
  wrong, so none is removed.

NOT removed, but DO NOT USE AS FEATURES:
- left_datasets / right_datasets  - which source lists the record came from.
  Predicts the label at ~92% accuracy on its own, without reading any name.
  It describes how the labels were made, not whether the entities match.
- left_topics / right_topics - why an entity was listed, not who it is.

Known label noise: 11 self-pairs were labelled negative (a record declared not
to be itself). Removed with the other self-pairs, but worth a line in the
report: the labels are analyst decisions, not ground truth.
"""
(out_dir / "README.md").write_text(note, encoding="utf-8")
print(f"wrote {out_dir / 'README.md'}")

# `pairs` and `scoped` are no longer needed: parts 4-8 work on `clean`.
# The proof cells in 3.0 are the last thing that reads them.
del scoped
gc.collect()
print("freed `scoped`; `clean` is the working table from here on")

---

# 4. What can we actually compute?

A feature needs its field present on **both** sides of a pair to be
computable for that pair. This part decides which features are even possible.

### 4.1 How often is each field present on both sides?

*The ceiling on every feature we might build.*

In [ ]:
PROPS = ["name", "alias", "firstName", "lastName", "middleName", "birthDate",
         "country", "nationality", "idNumber", "address", "topics"]

presence = pd.DataFrame([
    {
        "field": p,
        "left %":  round(100 * (clean[f"left_{p}"].map(len) > 0).mean(), 1),
        "right %": round(100 * (clean[f"right_{p}"].map(len) > 0).mean(), 1),
        "both %":  round(100 * ((clean[f"left_{p}"].map(len) > 0)
                                & (clean[f"right_{p}"].map(len) > 0)).mean(), 1),
    }
    for p in PROPS
]).sort_values("both %", ascending=False).set_index("field")

print("A feature needs the field on BOTH sides to be computable.\n")
print(presence.to_string())

presence["both %"].plot.barh(figsize=(7, 5), color="#8F73AC")
plt.title("Field present on both sides (%)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
clean.head()

In [ ]:
clean[clean["right_caption"].fillna("").str.strip() == ""]

### 4.2 Does field presence differ between the two classes?

*If it does, missingness is itself a signal. We decide whether that is legitimate or leakage.*

In [ ]:
rows = []
for p in PROPS:
    both = ((clean[f"left_{p}"].map(len) > 0)
            & (clean[f"right_{p}"].map(len) > 0))
    by = clean.assign(has=both).groupby("judgement").has.mean() * 100
    rows.append({"field": p,
                 "positive %": round(by.get("positive", 0), 1),
                 "negative %": round(by.get("negative", 0), 1),
                 "gap": round(by.get("positive", 0) - by.get("negative", 0), 1)})

gaps = (pd.DataFrame(rows).set_index("field")
        .reindex(pd.DataFrame(rows).set_index("field").gap.abs()
                 .sort_values(ascending=False).index))
print(gaps.to_string())

print("\nThese gaps are large, not marginal.")
print("Missingness predicts the label on its own, which makes it a leakage")
print("candidate: it describes which pairs an analyst was shown, not whether")
print("two entities are the same. Keep the indicators, but in part 7 measure")
print("what they contribute, and fit the model once with and once without.")

### 4.3 What form do birth dates take?

*Check precision and whether one record can carry several. Decides whether date agreement is a boolean or a multi-level category.*

In [ ]:
lengths = Counter(len(v) for lst in clean.left_birthDate for v in lst)
total = sum(lengths.values())

print("birthDate is stored as TEXT, at mixed precision:")
for n, label in [(10, "full date  1969-06-16"), (7, "year+month 1969-06"),
                 (4, "year only  1969")]:
    print(f"  {label:<24} {lengths.get(n, 0):>8,}  ({100*lengths.get(n,0)/total:.1f}%)")

multi = clean.left_birthDate.map(len) > 1
print(f"\nentities carrying more than one birth date: {100*multi.mean():.1f}%")
print("(different source lists disagreeing about the same person)")

print("\nExample of a record with several:")
ex = clean.loc[multi, "left_birthDate"].iloc[0]
print(" ", list(ex))

print("\nConsequence: '1969' vs '1969-06-16' is a PARTIAL match, not a conflict.")
print("Birth-date agreement must be four-way: exact / same year / conflict / missing.")

### 4.4 How many names does one entity carry?

*Decides whether to compare captions only or across the full alias lists.*

In [ ]:
n_names = clean.left_name.map(len)

print(f"names per entity: median {n_names.median():.0f}, "
      f"p90 {n_names.quantile(0.9):.0f}, max {n_names.max():,}")
print(f"entities with only one name: {100 * (n_names == 1).mean():.1f}%")
print(f"entities with more than five: {100 * (n_names > 5).mean():.1f}%")

n_names.clip(upper=30).plot.hist(bins=30, figsize=(7, 3), color="#C7952D")
plt.title("Names held per entity (capped at 30)")
plt.tight_layout()
plt.show()

print("\nAn entity with the most spellings:")
most = clean.loc[n_names.idxmax()]
print(f"  {most.left_caption}")
print(f"  {len(most.left_name)} spellings, first five: {list(most.left_name)[:5]}")

print("\nComparing captions only throws all the other spellings away.")
print("Whether that matters is question 4.5.")

### 4.5 Is comparing across alias lists worth the extra cost?

*Measure it before building it. More spellings means more chances of a coincidental match too.*

In [ ]:
sample = clean.sample(8_000, random_state=42)
y = (sample.judgement == "positive").astype(int)

def names_of(lst, caption):
    vals = [str(v) for v in lst][:12]          # cap: cost grows as n x m
    return vals or [str(caption)]

caption_score = np.array([fuzz.WRatio(a, b)
                          for a, b in zip(sample.left_caption, sample.right_caption)])

best_score = np.array([
    max((fuzz.WRatio(x, z) for x in names_of(ln, lc) for z in names_of(rn, rc)),
        default=0)
    for ln, rn, lc, rc in zip(sample.left_name, sample.right_name,
                              sample.left_caption, sample.right_caption)
])

auc_cap = roc_auc_score(y, caption_score)
auc_best = roc_auc_score(y, best_score)

print(f"AUC, captions only          : {auc_cap:.3f}")
print(f"AUC, best across all names  : {auc_best:.3f}")
print(f"gain                        : {auc_best - auc_cap:+.3f}")

print(f"\nmean lift on positives: {(best_score - caption_score)[y == 1].mean():+.1f} points")
print(f"mean lift on negatives: {(best_score - caption_score)[y == 0].mean():+.1f} points")
print("\nIt lifts negatives too: more spellings means more chances of a")
print("coincidental match. Cheap enough to keep, but it will not carry the model.")

---

# 5. Where is the difficulty?

This is the part that produces the findings worth presenting.

### 5.1 How well does a simple name-similarity score separate the two classes?

*Plot both distributions on the same axes. The region where they overlap is the whole problem.*

In [ ]:
sample = clean.sample(60_000, random_state=42).copy()
sample["sim"] = [fuzz.WRatio(a, b)
                 for a, b in zip(sample.left_caption, sample.right_caption)]
y = (sample.judgement == "positive").astype(int)

fig, ax = plt.subplots(figsize=(8, 4))
for label, colour in [("positive", "#2a7f62"), ("negative", "#b4474e")]:
    ax.hist(sample.loc[sample.judgement == label, "sim"],
            bins=50, alpha=0.6, density=True, label=label, color=colour)
ax.set_xlabel("name similarity (RapidFuzz WRatio)")
ax.set_ylabel("density")
ax.set_title("Same entity vs different entity: the scores overlap")
ax.legend()
plt.tight_layout()
plt.show()

print(sample.groupby("judgement").sim.describe()[["25%", "50%", "75%"]].round(1).to_string())
print(f"\nAUC of this single score: {roc_auc_score(y, sample.sim):.3f}")
print(f"pairs in the ambiguous 60-95 band: "
      f"{100 * ((sample.sim > 60) & (sample.sim < 95)).mean():.1f}%")
print(f"different entities scoring a perfect 100: "
      f"{100 * (sample.loc[sample.judgement == 'negative', 'sim'] == 100).mean():.2f}%")
print("\nThe overlap region IS the problem. No threshold on this score splits")
print("it without making one kind of mistake or the other.")

### 5.2 Are there pairs with identical names that are different entities?

*If so, no string metric can ever separate them. Check whether any other field can.*

In [ ]:
identical = (clean.left_caption.str.strip().str.casefold()
             == clean.right_caption.str.strip().str.casefold())
neg_identical = clean[identical & (clean.judgement == "negative")]

print(f"pairs with identical names      : {identical.sum():,}")
print(f"  of which DIFFERENT entities   : {len(neg_identical):,} "
      f"({100 * len(neg_identical) / identical.sum():.1f}%)")
print("\nNo string metric can ever separate these. They are a hard floor under")
print("the error rate of any name-only approach.")

def bd_state(left, right):
    L, R = set(left), set(right)
    if not L or not R:                              return "missing"
    if L & R:                                       return "exact"
    if {v[:4] for v in L} & {v[:4] for v in R}:     return "same year"
    return "conflict"

states = pd.Series([bd_state(l, r) for l, r in
                    zip(neg_identical.left_birthDate, neg_identical.right_birthDate)])
print("\nCan a birth date rescue them?")
print(states.value_counts().to_string())
have = 100 * (states != "missing").mean()
print(f"\nOnly {have:.0f}% carry a birth date on both sides.")
print(f"For the other {100-have:.0f}% NOTHING in this dataset separates them.")
print("\nThat is the operational finding: the fix is not a better algorithm,")
print("it is collecting a date of birth at onboarding.")

print("\nTen of them:")
print(neg_identical[["left_caption", "right_caption", "left_schema", "right_schema"]].head(10).to_string(index=False))

### 5.3 Are company and organisation pairs different from person pairs?

*Company names carry legal suffixes shared by unrelated firms. Check whether the two behave the same.*

In [ ]:
sample["kind"] = np.where(sample.left_schema == "Person", "Person", "Company/Org")

rows = []
for kind, g in sample.groupby("kind"):
    rows.append({
        "kind": kind,
        "pairs": f"{len(g):,}",
        "% positive": round(100 * (g.judgement == "positive").mean(), 1),
        "median score, positives": round(g.loc[g.judgement == "positive", "sim"].median(), 1),
        "median score, negatives": round(g.loc[g.judgement == "negative", "sim"].median(), 1),
        "AUC": round(roc_auc_score((g.judgement == "positive").astype(int), g.sim), 3),
    })
print(pd.DataFrame(rows).set_index("kind").to_string())

print("\nFor companies the name score is barely better than a coin toss, and the")
print("median score for TRUE matches is lower than for non-matches.")
print("\nWhy: legal suffixes. The commonest tokens in the whole database are")
print("shared by millions of unrelated firms:")
print(freq_tokens.head(12).to_string())
print("\nStrip them before scoring, and never report people and companies as")
print("one number.")

### 5.4 What happens on pairs written in two different alphabets?

*Measure the score's behaviour there rather than assuming. Then test whether simple normalisation helps.*

In [ ]:
def script_of(text):
    """Rough writing system, from the first few letters."""
    names = []
    for ch in str(text):
        if not ch.isalpha():
            continue
        try:
            names.append(unicodedata.name(ch).split()[0])
        except ValueError:
            names.append("UNKNOWN")
        if len(names) >= 8:
            break
    if not names:
        return "none"
    return max(set(names), key=names.count).title()

sample["ls"] = sample.left_caption.map(script_of)
sample["rs"] = sample.right_caption.map(script_of)
cross = sample.ls != sample.rs

print(f"pairs mixing two writing systems: {100 * cross.mean():.1f}%")
print("\nscript of the left-hand name:")
print((sample.ls.value_counts(normalize=True).head(5) * 100).round(2).to_string())

print("\nscript of the right-hand name:")
print((sample.rs.value_counts(normalize=True).head(5) * 100).round(2).to_string())

auc_same = roc_auc_score((sample.loc[~cross, "judgement"] == "positive").astype(int),
                         sample.loc[~cross, "sim"])
auc_cross = roc_auc_score((sample.loc[cross, "judgement"] == "positive").astype(int),
                          sample.loc[cross, "sim"])
print(f"\nAUC on same-script pairs : {auc_same:.3f}")
print(f"AUC on cross-script pairs: {auc_cross:.3f}")

# does simple normalisation help?
def strip_accents(s):
    s = unicodedata.normalize("NFKD", str(s))
    return "".join(c for c in s if not unicodedata.combining(c))

cs = sample[cross]
sim_t = [fuzz.WRatio(strip_accents(a), strip_accents(b))
         for a, b in zip(cs.left_caption, cs.right_caption)]
auc_t = roc_auc_score((cs.judgement == "positive").astype(int), sim_t)
print(f"after stripping accents  : {auc_t:.3f}")

print("\nBelow 0.5 means the score points the WRONG way there: any resemblance")
print("between a Cyrillic and a Latin string is coincidence. Accent-stripping")
print("does not bridge alphabets. Real transliteration is a step-4 decision;")
print("for now add a cross_script flag so the model can distrust the score.")

### 5.5 Does how common a name is relate to difficulty?

*Use names.txt for frequency. This is what the second dataset is for; confirm it earns its place.*

In [ ]:
def last_token_freq(caption):
    toks = str(caption).casefold().split()
    return freq_tokens.get(toks[-1], 0) if toks else 0

sample["name_freq"] = sample.left_caption.map(last_token_freq)
sample["band"] = pd.cut(sample.name_freq, [-1, 0, 10, 100, 1000, 10**9],
                        labels=["unknown", "rare (1-10)", "uncommon (11-100)",
                                "common (101-1k)", "very common (>1k)"])

ambiguous = (sample.sim > 60) & (sample.sim < 95)
cats = list(sample.band.cat.categories)
table = pd.DataFrame({
    "pairs": sample.band.value_counts().reindex(cats),
    "% ambiguous": (sample.assign(a=ambiguous).groupby("band", observed=False)
                    .a.mean().reindex(cats) * 100).round(1),
    "% negative": (sample.assign(n=sample.judgement == "negative")
                   .groupby("band", observed=False).n.mean().reindex(cats) * 100).round(1),
})
print(table.to_string())

print("\nThe trend is monotone: the commoner the surname, the more likely a")
print("candidate pair is a coincidence. Name frequency earns its place as a")
print("feature, and this is what the second dataset is for.")
print("\nCaveat for the report: we approximate 'surname' by the last token,")
print("which is wrong for several naming conventions and for organisations.")

# Zipf: the reason rarity carries so much information
plt.figure(figsize=(6, 4))
plt.loglog(range(1, len(freq_tokens) + 1), freq_tokens.values)
plt.xlabel("rank"); plt.ylabel("frequency")
plt.title("Token frequency follows a power law")
plt.tight_layout()
plt.show()

---

# 6. Is this one problem or two?

### 6.1 Split everything by whether a birth date exists on both sides

*Recompute the balance, the score distributions and the difficulty separately for each half.*

In [ ]:
clean["bd"] = [bd_state(l, r) for l, r in
               zip(clean.left_birthDate, clean.right_birthDate)]

table = pd.crosstab(clean.bd, clean.judgement)
pct = (table.div(table.sum(axis=1), axis=0) * 100).round(1)
pct["rows"] = table.sum(axis=1)
print(pct.to_string())

print("\nRead the 'missing' row: the classes are nearly even there. That is the")
print("half of the data where names are all you have.")

sample["bd"] = [bd_state(l, r) for l, r in
                zip(sample.left_birthDate, sample.right_birthDate)]
sample["half"] = np.where(sample.bd == "missing", "no birth date", "birth date present")

rows = []
for half, g in sample.groupby("half"):
    rows.append({"half": half, "pairs": f"{len(g):,}",
                 "% positive": round(100 * (g.judgement == "positive").mean(), 1),
                 "AUC of name score": round(roc_auc_score(
                     (g.judgement == "positive").astype(int), g.sim), 3)})
print("\n" + pd.DataFrame(rows).set_index("half").to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (half, g) in zip(axes, sample.groupby("half")):
    for label, colour in [("positive", "#2a7f62"), ("negative", "#b4474e")]:
        ax.hist(g.loc[g.judgement == label, "sim"], bins=40, alpha=.6,
                density=True, label=label, color=colour)
    ax.set_title(half); ax.set_xlabel("name similarity")
axes[0].set_ylabel("density"); axes[0].legend()
fig.suptitle("The same score, two different problems")
plt.tight_layout()
plt.show()

### 6.2 If one half is harder, is that the metric's fault or the data's?

*Try several different string metrics on the same split. If they all agree, it is the data.*

In [ ]:
metrics = {
    "WRatio":           fuzz.WRatio,
    "ratio":            fuzz.ratio,
    "token_sort_ratio": fuzz.token_sort_ratio,
    "token_set_ratio":  fuzz.token_set_ratio,
    "partial_ratio":    fuzz.partial_ratio,
    "Jaro-Winkler":     lambda a, b: rf_distance.JaroWinkler.similarity(a, b) * 100,
}

easy = sample[sample.bd != "missing"]
hard = sample[sample.bd == "missing"]

rows = []
for name, fn in metrics.items():
    rows.append({
        "metric": name,
        "AUC, birth date": round(roc_auc_score(
            (easy.judgement == "positive").astype(int),
            [fn(a, b) for a, b in zip(easy.left_caption, easy.right_caption)]), 3),
        "AUC, no birth date": round(roc_auc_score(
            (hard.judgement == "positive").astype(int),
            [fn(a, b) for a, b in zip(hard.left_caption, hard.right_caption)]), 3),
    })
print(pd.DataFrame(rows).set_index("metric").to_string())
print("\nEvery metric lands in the same place, so it is the data, not the metric.")

hard_person = hard[hard.left_schema == "Person"]
auc_hp = roc_auc_score((hard_person.judgement == "positive").astype(int), hard_person.sim)
print(f"\nPerson pairs with no birth date: AUC {auc_hp:.3f}  (n={len(hard_person):,})")
print("Below 0.5. A MORE similar name makes the pair MORE likely to be two")
print("different people.")

# Does token structure survive where character matching does not?
def tokens(s):
    return set(str(s).casefold().split())

jacc = [len(tokens(a) & tokens(b)) / max(len(tokens(a) | tokens(b)), 1)
        for a, b in zip(hard_person.left_caption, hard_person.right_caption)]
print(f"\ntoken Jaccard on the same pairs : AUC "
      f"{roc_auc_score((hard_person.judgement == 'positive').astype(int), jacc):.3f}")
print("\nSo it is CHARACTER-level matching that fails, not names. The candidate")
print("generator already spent the character signal when it chose which pairs")
print("to propose; token structure survives because it never used it.")

### 6.3 Decide: one model or two?

*Write the decision and the reason. This shapes steps 4 and 5 of the project.*

In [ ]:
print("""DECISION: one model, two reports.

Birth-date agreement enters as a four-level categorical feature
(exact / same year / conflict / missing), not a boolean and not a split.

Why one model:
  - two models halve the training data for each
  - two models mean two thresholds, two feature sets, two error taxonomies,
    maintained by two people in a course project
  - a tree model will learn the split by itself; the signal is enormous

Why two reports:
  - the two halves are genuinely different problems (6.1, 6.2)
  - a single combined cost curve would hide that character matching is
    near-useless on half the traffic
  - the operational recommendation falls out of the comparison

What we do NOT do: report one combined number.""")

---

# 7. Leakage check

The last thing before feature engineering, and the easiest to skip. Some
columns describe how the dataset was assembled rather than who the entities
are. A model given those looks excellent and means nothing.

### 7.0 Prove the leakage mechanism, not only correlation.

*High AUC is a symptom. We can exclude the column only by understanding WHY.*

In [ ]:
# HYPOTHESIS: some columns leak - they describe how the dataset was built,
# not who the entities are. Before excluding anything, prove the mechanism.

from sklearn.metrics import roc_auc_score
y = (clean.judgement == "positive").astype(int).values

# --- the claim: records from the SAME source list are usually DIFFERENT people
shared = np.array([len(set(a) & set(b)) > 0
                   for a, b in zip(clean.left_datasets, clean.right_datasets)])
print("do the two records share a source list?")
print(pd.crosstab(shared, clean.judgement, normalize="index").round(3).to_string())
print(f"\nAUC using ONLY this, reading no name: "
      f"{roc_auc_score(y, (~shared).astype(float)):.3f}")

print("""
WHY. Deduplication merges records ACROSS source lists, and one list does not
normally contain the same entity twice. So "same list" implies "different
entity" almost mechanically. The column describes the labelling process.
""")

# --- a feature that leaks the other way
n_names = clean.left_name.map(len).values.astype(float)
auc_names = roc_auc_score(y, n_names)
print(f"AUC of 'how many names the left record holds': {auc_names:.3f}")
print(f"  strength = max(AUC, 1-AUC) = {max(auc_names, 1-auc_names):.3f}, "
      f"pointing at {'same' if auc_names > .5 else 'different'} entity")
print("Same mechanism: a merged entity accumulates spellings.")

# --- the control: a feature that genuinely describes the entities
from rapidfuzz import fuzz
chk = clean.sample(20_000, random_state=0)
name_score = [fuzz.WRatio(a, b) for a, b in zip(chk.left_caption, chk.right_caption)]
print(f"\nfor comparison, an honest feature (name similarity): "
      f"AUC {roc_auc_score((chk.judgement=='positive').astype(int), name_score):.3f}")
print("\nThe leaky columns beat the thing we are actually trying to measure.")
print("That is the tell.")

### 7.1 Can anything predict the label without reading the names at all?

*Test each suspect column on its own. Watch for features that point the *wrong* way: they are just as predictive.*

In [ ]:
y = (clean.judgement == "positive").astype(int).values
candidates = {}

shared_source = np.array([len(set(a) & set(b)) > 0
                          for a, b in zip(clean.left_datasets, clean.right_datasets)])
candidates["shares a source list"] = (~shared_source).astype(float)
candidates["number of source lists (left)"] = -clean.left_datasets.map(len).values.astype(float)
# id type: workflow metadata, measured here so the exclusion is evidenced
import re as _re
def _kind(v):
    v = str(v)
    return 1.0 if (v.startswith("NK-") or _re.match(r"^Q\d+$", v)) else 0.0

lk = clean.left_id.map(_kind).values
rk = clean.right_id.map(_kind).values
candidates["both sides are merged entities"] = (lk * rk)
candidates["exactly one side is a merged entity"] = (lk != rk).astype(float)
candidates["number of merged sides (0, 1 or 2)"] = lk + rk
candidates["number of names held (left)"] = clean.left_name.map(len).values.astype(float)
candidates["left is sanctioned (topics)"] = clean.left_topics.map(
    lambda t: "sanction" in t).values.astype(float)
candidates["schemas identical"] = (clean.left_schema == clean.right_schema).values.astype(float)
for prop in ("country", "address", "idNumber", "birthDate"):
    candidates[f"{prop} present both sides"] = (
        (clean[f"left_{prop}"].map(len) > 0)
        & (clean[f"right_{prop}"].map(len) > 0)).values.astype(float)

rows = []
for name, score in candidates.items():
    auc = roc_auc_score(y, score)
    rows.append({"candidate": name,
                 "AUC": round(auc, 3),
                 "strength": round(max(auc, 1 - auc), 3),
                 "points to": "same entity" if auc >= 0.5 else "different entity"})

leak = pd.DataFrame(rows).sort_values("strength", ascending=False).set_index("candidate")
print("None of these reads a name.\n")
print(leak.to_string())

print("\n'strength' is max(AUC, 1-AUC): a feature at AUC 0.18 is exactly as")
print("predictive as one at 0.82, it just points the other way. Sorting on raw")
print("AUC would have hidden half of them.")

### 7.2 Which candidates must be excluded, and which are borderline?

*For each one, say what mechanism makes it leaky. If we cannot explain the mechanism, we have not found a leak yet.*

In [ ]:
print("""EXCLUDE - these describe how the data was assembled:

  left_datasets / right_datasets
      Deduplication merges records ACROSS source lists, and one list does not
      normally contain the same entity twice. So "same list" implies "different
      entity" almost mechanically. Predicts the label at ~92% with no name.

  number of source lists held
      The same signal in another shape: an entity on many lists was merged
      from many lists.

  topics
      Says why an entity was listed (sanction, PEP, crime), not who it is.
      Two unrelated sanctioned people share it; it carries no identity signal.

  number of names held
      Same mechanism again: a merged entity accumulates spellings.

KEEP, BUT MEASURE SEPARATELY - field-presence indicators:

  country / address / idNumber / birthDate present on both sides
      These predict the label well above chance. Partly legitimate: a real
      screening system also knows which fields it holds, and a missing field
      genuinely changes how confident you can be. Partly an artefact of which
      pairs an analyst was shown.
      We cannot separate the two parts, so: fit the model once with these and
      once without, report both, and say plainly that the difference is an
      upper bound on how much of our performance comes from the sampling
      process rather than from the entities.

THE TEST we used, for the report:
  a column is process-describing if you could fill it in without knowing
  anything about the person.""")

---

# 8. Wrap up

### 8.0a What must the split guarantee, and what counts as one entity?

*The split decides whether every later number means anything, and it cannot be repaired afterwards. The only real decision is which rows count as a link between two ids.*


In [ ]:
# The split must guarantee one thing: no id appears in two different splits.
# If it does, the model memorises that id while training and meets it again at
# test time, and the test number stops measuring generalisation and starts
# measuring memory.
#
# That rules out throwing rows into splits independently. Ids have to be
# grouped first, and whole groups sent to one split. The grouping rule is the
# decision, and it reduces to a single question: which rows count as a link
# between two ids?
#
#   Option A - every row is a link.
#       Reads as: "these two ids were compared to each other."
#       A group is then a pile of ids that happened to get compared, which is
#       not an entity.
#
#   Option B - only positive rows are links.
#       Reads as: "an analyst decided these two ids are the same entity."
#       Chains count: A=B and B=C means A, B, C are one entity.
#       A group is then an entity, which is exactly the thing we must not
#       split across train and test.
#
# Both are measured here, because the difference turns out to be large.

from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

left = clean.left_id.astype(str)
right = clean.right_id.astype(str)
entities = pd.Index(pd.unique(pd.concat([left, right])))
row_of = {e: i for i, e in enumerate(entities)}
li = left.map(row_of).to_numpy()
ri = right.map(row_of).to_numpy()
n_ent = len(entities)
is_pos = (clean.judgement == "positive").to_numpy()

def group_ids(link_mask):
    """Group ids, using only the rows selected by link_mask as links."""
    graph = coo_matrix((np.ones(int(link_mask.sum())),
                        (li[link_mask], ri[link_mask])),
                       shape=(n_ent, n_ent))
    return connected_components(graph, directed=False)

# Which split a group goes to is decided by a hash of the group's smallest id,
# not by a random generator. With a seed, the groups draw numbers in order,
# so removing a single group shifts every number after it and reshuffles the
# whole test set. A hash depends only on the group itself: a group keeps its
# split however the rest of the cleaning changes.
import hashlib

SPLIT_SALT = "split-v1"     # change only on purpose: it reassigns every group

def unit(key, salt=SPLIT_SALT):
    """Map a string to a fixed number in [0, 1)."""
    return int(hashlib.sha1(f"{salt}|{key}".encode()).hexdigest()[:12], 16) / 16**12

def assign_split(n_groups, group_of):
    """60 / 20 / 20 by group, keyed on each group's smallest id."""
    first_id = pd.Series(entities).groupby(group_of).min().reindex(range(n_groups))
    u = np.array([unit(k) for k in first_id])
    return np.where(u < 0.6, "train", np.where(u < 0.8, "val", "test"))

def evaluate(name, link_mask):
    n_groups, group_of = group_ids(link_mask)
    sizes = pd.Series(group_of).value_counts()

    where = assign_split(n_groups, group_of)
    ls, rs = where[group_of[li]], where[group_of[ri]]
    keep = ls == rs

    print(f"--- {name}")
    print(f"    groups                : {n_groups:,}")
    print(f"    group size            : median {sizes.median():.0f}, "
          f"95th {sizes.quantile(0.95):.0f}, max {sizes.max():,} "
          f"({100*sizes.max()/n_ent:.2f}% of ids)")
    print(f"    rows kept             : {keep.sum():,} / {len(clean):,} "
          f"({100*keep.mean():.1f}%)")
    dropped_pos = int((is_pos & ~keep).sum())
    print(f"    rows dropped          : {int((~keep).sum()):,} "
          f"(positive {dropped_pos:,}, negative {int((~keep).sum())-dropped_pos:,})")
    parts = []
    for s in ("train", "val", "test"):
        sel = keep & (ls == s)
        share = 100 * is_pos[sel].mean() if sel.sum() else float("nan")
        parts.append(f"{s} {100*sel.sum()/max(keep.sum(),1):.0f}%/pos {share:.0f}%")
    print(f"    split size / balance  : " + "   ".join(parts))
    return n_groups, group_of

all_rows = np.ones(len(clean), dtype=bool)
print("Option A, every row is a link:\n")
nA, gA = evaluate("every row", all_rows)
print("\nOption B, only positive rows are links:\n")
nB, gB = evaluate("positive rows only", is_pos)

print("""
Reading: Option A keeps every row, because a row can never straddle two
groups when the row itself is a link. The cost is not lost rows, it is that
one group swallows roughly a quarter of all ids, lands whole in one split,
and wrecks both the split sizes and the label balance. Option B loses the
rows that connect two different entities, every one of them a negative, and
in exchange its groups are entity-sized and the splits are usable.
""")


### 8.0b Why does treating every row as a link produce one enormous group?

*Understand the mechanism before working around it, because the same mechanism will appear again in feature engineering.*


In [ ]:
# Option A produced one group holding about a quarter of all ids. That is not
# an accident of this dataset, it is what negative rows do to a graph.
#
# A negative row says "these two are different". Used as a link it still joins
# them. A record with a common name gets compared against many different
# candidates, so it becomes a hub; two hubs sharing any candidate merge; and
# the merges cascade until a large share of the data is one blob.

deg = pd.Series(np.concatenate([li, ri])).value_counts()
deg_pos = pd.Series(np.concatenate([li[is_pos], ri[is_pos]])).value_counts()

print("rows an id takes part in (all rows vs positive rows only):")
print(pd.DataFrame({
    "all rows": deg.describe(percentiles=[0.5, 0.99]).round(1),
    "positive only": deg_pos.describe(percentiles=[0.5, 0.99]).round(1),
}).to_string())

busiest = deg.head(5).index
print("\nthe five busiest ids, and how their rows are labelled:")
for pos_i in busiest:
    ident = entities[pos_i]
    touches = (li == pos_i) | (ri == pos_i)
    n_pos = int((touches & is_pos).sum())
    cap = clean.loc[(clean.left_id.astype(str) == ident), "left_caption"]
    if cap.empty:
        cap = clean.loc[(clean.right_id.astype(str) == ident), "right_caption"]
    name = cap.iloc[0] if len(cap) else "?"
    print(f"   {ident:<28} rows {int(touches.sum()):>4}  "
          f"positive {n_pos:>4}  negative {int(touches.sum())-n_pos:>4}  {name[:38]}")

sizes_A = pd.Series(gA).value_counts()
sizes_B = pd.Series(gB).value_counts()
print(f"\nlargest group: {sizes_A.max():,} ids when every row links, "
      f"{sizes_B.max():,} ids when only positive rows link")
print("Negative rows are what glue unrelated entities together, so they must")
print("not be used as links. They are still kept as training rows.")


### 8.0c Build and save the split

*Groups come from positive rows only. Whole groups go to one split. Rows crossing two groups are dropped and counted.*


In [ ]:
# Build the split we will actually use: groups from positive rows only.

GROUP_LINKS = "positive"      # "positive" (chosen) or "all" (see 8.0a)

link_mask = is_pos if GROUP_LINKS == "positive" else np.ones(len(clean), bool)
n_groups, group_of = group_ids(link_mask)

group_split = assign_split(n_groups, group_of)     # hash-based, see 8.0a

ls, rs = group_split[group_of[li]], group_split[group_of[ri]]
keep = ls == rs

split_df = clean[keep].copy()
split_df["split"] = ls[keep]
split_df["entity_group"] = group_of[li][keep]

print(f"entities (groups)   : {n_groups:,}")
print(f"rows kept           : {len(split_df):,} / {len(clean):,} "
      f"({100*keep.mean():.1f}%)")
print(f"rows dropped        : {int((~keep).sum()):,}, of which positive "
      f"{int((is_pos & ~keep).sum()):,}")
print()
summary = (split_df.assign(positive=split_df.judgement.eq("positive"))
           .groupby("split")
           .agg(rows=("judgement", "size"),
                positive_share=("positive", "mean"),
                entities=("entity_group", "nunique")))
summary["share_of_rows"] = summary.rows / summary.rows.sum()
print(summary.to_string(float_format=lambda v: f"{v:.3f}"))

# The guarantee, asserted rather than assumed.
grp = pd.DataFrame({"group": np.concatenate([group_of[li][keep],
                                             group_of[ri][keep]]),
                    "split": np.concatenate([split_df.split.to_numpy(),
                                             split_df.split.to_numpy()])})
leaked = grp.groupby("group").split.nunique()
assert (leaked == 1).all(), "a group appears in more than one split"
print("\ncheck passed: no entity group appears in more than one split")

# Known remaining property, stated rather than hidden: the label balance is
# not equal across splits. Dropped rows are all negative, and a negative row
# survives far more often inside train (0.6 x 0.6) than inside test
# (0.2 x 0.2), so train retains proportionally more negatives.
# What this does and does not affect:
#   - the false-alarm RATE (false alarms / negatives) and recall do not
#     depend on the share of positives in a split, so they compare fairly;
#   - precision and "false alarms per N pairs" do depend on it.
# So we report the false-alarm rate together with the recall actually
# achieved, and rescale any per-pair figure to one fixed positive share.

SPLIT_PATH = C.PAIRS_SPLIT
SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)
split_df.to_parquet(SPLIT_PATH, index=False)
print(f"\nwritten: {SPLIT_PATH}  ({len(split_df):,} rows)")


### 8.1 Collect the decisions this notebook produced

*One line each: what was decided, and the number that decided it.*

In [ ]:
decisions = pd.DataFrame([
    ("3.1", "Scope", "Keep Person / Company / Organization, filtered on BOTH sides",
     f"{n_raw:,} -> {n_scoped:,}"),
    ("3.2", "Cleaning", "Remove self-pairs, then duplicate id-pairs (order matters)",
     f"-{n_self:,} self, -{n_dups:,} dup"),
    ("3.2", "Cleaning", "Remove every pair recorded with both labels (label unknown)",
     f"-{n_conflict_pairs:,} pairs ({n_conflict_rows:,} rows)"),
    ("3.2b", "Cleaning", "Keep negatives that contradict a chain of positives; count them",
     f"{n_neg_in_group:,} kept"),
    ("3.3", "Balance", "Report the cleaned balance, not the raw one",
     f"{100*(clean.judgement=='positive').mean():.1f}% positive"),
    ("2.4", "Split", "Split train/val/test BY ENTITY, never by row",
     "most rows touch a reused entity"),
    ("4.3", "Birth date", "Four-level agreement, not boolean", "24% are year-only"),
    ("5.3", "Record type", "Never one metric across people and companies", "company AUC ~0.54"),
    ("5.4", "Script", "Add a cross_script flag; transliteration is step 4", "cross-script AUC < 0.5"),
    ("6.3", "Model", "One model, two reports", "split on birth-date availability"),
    ("7.2", "Leakage", "Exclude datasets, topics, name counts; measure presence flags",
     "source overlap ~92%"),
    ("8.0c", "Split", "Split assigned by a hash of each group's smallest id, not a seed",
     "a group keeps its split when the cleaning changes"),
    ("-", "Paths", "All paths from src/screening/config.py; notebook moved to notebooks/",
     "PAIRS_INPUT = PAIRS_CLEAN"),
], columns=["question", "topic", "decision", "evidence"])

print(decisions.to_string(index=False))

decisions.to_csv(project_root / "docs/eda_decisions.csv", index=False)
print(f"\nwrote {project_root / 'docs/eda_decisions.csv'}")

### 8.2 Save the final prepared dataset and note what step 4 needs from it

*Feature engineering starts from here.*

In [ ]:
out_path = C.PAIRS_CLEAN

# Read metadata and one column, not the whole file: `pairs`, `scoped` and
# `clean` are all still in memory and a fourth full copy will run you out.
meta = pq.read_metadata(out_path)
judgement = pd.read_parquet(out_path, columns=["judgement"]).judgement

print(f"{out_path.name}: {meta.num_rows:,} rows x {meta.num_columns} columns")
print(f"balance: {100 * (judgement == 'positive').mean():.1f}% positive")
print(f"size   : {out_path.stat().st_size / 1e6:.0f} MB")
print("\nIf memory is tight, `del pairs, scoped` once the file is written.")

print("""
WHAT STEP 4 NEEDS FROM THIS FILE

Build first, before any features:
  an entity-level train / validation / test splitter. Group by entity id, then
  divide the GROUPS 60/20/20. The threshold is chosen on validation and the
  test set is read once, at the end.

Feature families, in order of expected value:
  1. token-based   token overlap, Jaccard, IDF-weighted shared tokens
                   (these survive where character matching fails - see 6.2)
  2. record-based  birth-date agreement (4 levels), country overlap,
                   nationality, idNumber match
  3. name rarity   n_freq_* in name_features.py
  4. character     WRatio, Jaro-Winkler, edit distance
                   (weakest family; include for comparison, not as the core)
  5. flags         cross_script, is_company, missing-data indicators

Do NOT build: anything from datasets, topics, or counts of names held.

Report everything split by birth-date availability.
""")

---
---

# Appendix: earlier work

## A1. Earlier exploration

In [ ]:
display(df.head(10))

df.info()

print(df.columns.tolist())

In [ ]:
df.isna().sum

In [ ]:
df["judgement"].value_counts()

df['left_schema'].value_counts()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left schema
df['left_schema'].value_counts().plot(kind='bar', ax=axes[0])

axes[0].set(
    title='Distribution of left schemas',
    xlabel='Category',
    ylabel='Count'
)

axes[0].bar_label(axes[0].containers[0])

# Right schema
df['right_schema'].value_counts().plot(kind='bar', ax=axes[1])

axes[1].set(
    title='Distribution of right schemas',
    xlabel='Category',
    ylabel='Count'
)

axes[1].bar_label(axes[1].containers[0])

plt.tight_layout()
plt.show()

- We have discrepancies across schema categories, (`right_schema` and `left_schema`)

In [ ]:
#or:

df.loc[
    df['left_schema'] != df['right_schema'],
    ['left_schema', 'right_schema']
].value_counts()

In [ ]:
df[['left_datasets','right_datasets']].head(20)

# 2. Can we trust the labels?

## 2.1 Are there duplicate pairs?

1. Count duplicate ids on `right_id` and `left_id`

In [ ]:
df.loc[:,['right_id','left_id']]

In [ ]:
df.loc[
    df['right_id']==df['left_id'],
    ['right_id','left_id']
].value_counts()

In [ ]:
id_duplicates = df.loc[
    df['right_id']==df['left_id'],
    ['right_id']].count()

id_duplicates/df.count() * 100